# Fine-Tuning Gemma 2 2B IT → Apex AI (GGUF portável, SEM Ollama)

Notebook para treinar o modelo Apex AI com:
- **Modelo final portável em GGUF**
- **Publicação no Hugging Face Hub** (automático)
- **Deploy como Inference Endpoint** na nuvem
- **Execução via API própria** — sem Ollama, sem lock-in

## Fluxo
1. Configurar ambiente (GPU + libs)
2. Carregar treino/teste da Apex
3. (Opcional) Expandir dataset com skills
4. Treinar LoRA em Gemma 2 2B IT
5. Avaliar em teste separado
6. Fundir + converter para GGUF
7. **Publicar no Hugging Face Hub**
8. **Gerar endpoint de inferência na nuvem**

## 1) Pre-check GPU
No Colab: Runtime -> Change runtime type -> T4 GPU

In [ ]:
import torch
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
    print('VRAM GB:', round(torch.cuda.get_device_properties(0).total_memory / 1e9, 2))
else:
    print('GPU nao detectada. Ative T4 no Colab antes de treinar.')

## 2) Instalacao de dependencias

In [ ]:
import subprocess, sys

def pip_install(packages):
    cmd = [sys.executable, '-m', 'pip', 'install', '-q', '-U'] + packages
    subprocess.run(cmd, check=False)

pip_install([
    'transformers',
    'datasets',
    'accelerate',
    'peft',
    'bitsandbytes',
    'trl',
    'sentencepiece',
    'protobuf'
])
print('Dependencias instaladas.')

## 3) Aceleracao opcional de DataFrame (GPU)
Ative somente se o ambiente tiver cudf instalado.

In [ ]:
try:
    get_ipython().run_line_magic('load_ext', 'cudf.pandas')
    print('cudf.pandas ativado.')
except Exception:
    print('cudf.pandas indisponivel. Seguindo com pandas normal.')

import pandas as pd

## 4) Carregar dataset Apex (treino/teste)
Esperado:
- apex_train.jsonl
- apex_test.jsonl

Fonte padrao: training_data do repo. Se falhar, faz upload manual.

In [ ]:
import os, json

REPO_RAW = 'https://raw.githubusercontent.com/jedgard70/apex-ai-copilot-platform/main/training_data'

def load_jsonl(path):
    if not os.path.exists(path) or os.path.getsize(path) == 0:
        return []
    rows = []
    with open(path, 'r', encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            if line:
                rows.append(json.loads(line))
    return rows

os.system(f"wget -q {REPO_RAW}/apex_train.jsonl -O apex_train.jsonl")
os.system(f"wget -q {REPO_RAW}/apex_test.jsonl -O apex_test.jsonl")

train_raw = load_jsonl('apex_train.jsonl')
test_raw = load_jsonl('apex_test.jsonl')

if not train_raw:
    print('Download automatico falhou. Faca upload manual de apex_train.jsonl e apex_test.jsonl.')
    try:
        from google.colab import files
        files.upload()
    except Exception as e:
        print('Upload manual nao disponivel neste ambiente:', e)
    train_raw = load_jsonl('apex_train.jsonl')
    test_raw = load_jsonl('apex_test.jsonl')

print('Treino:', len(train_raw))
print('Teste:', len(test_raw))
if train_raw:
    print('Exemplo treino keys:', list(train_raw[0].keys()))

## 5) Expansao opcional com skills da Apex (todas)
Se o repo estiver montado no ambiente, esta celula transforma SKILL.md em exemplos extras de instrucao-resposta.

Observacao: isso complementa o dataset principal; nao substitui exemplos reais de conversa.

In [ ]:
import glob
import os
import re
from collections import Counter


def normalize_ws(text: str) -> str:
    return re.sub(r"\s+", " ", str(text or "")).strip()


def find_repo_root() -> str:
    """Resolve repo root robustly for local/Colab execution."""
    candidates = []

    # Current working dir and parents
    cwd = os.getcwd()
    candidates.append(cwd)
    parent = cwd
    for _ in range(6):
        parent = os.path.dirname(parent)
        if parent and parent not in candidates:
            candidates.append(parent)

    # Common Colab paths
    for p in [
        "/content/apex-ai-copilot-platform",
        "/content",
        "/kaggle/working",
    ]:
        if p not in candidates:
            candidates.append(p)

    for base in candidates:
        if not base or not os.path.isdir(base):
            continue
        has_markers = any(
            os.path.exists(os.path.join(base, marker))
            for marker in ["AGENTS.md", "package.json", "CHECKPOINT_TRACKER.md"]
        )
        if has_markers:
            return base

    return cwd


def extract_skill_paths_from_agents_md(agents_md_path: str):
    """Parse explicit SKILL.md paths listed in AGENTS.md (<file>...</file>)."""
    results = []
    if not os.path.exists(agents_md_path):
        return results

    try:
        with open(agents_md_path, "r", encoding="utf-8", errors="ignore") as f:
            txt = f.read()
        for m in re.finditer(r"<file>(.*?)</file>", txt, flags=re.DOTALL):
            p = normalize_ws(m.group(1))
            if p.lower().endswith("skill.md") and os.path.exists(p):
                results.append(os.path.normpath(p))
    except Exception:
        pass

    return results


def discover_skill_files(repo_root: str):
    patterns = [
        os.path.join(repo_root, "skills", "**", "SKILL.md"),
        os.path.join(repo_root, ".agents", "skills", "**", "SKILL.md"),
        os.path.join(repo_root, ".github", "**", "skills", "**", "SKILL.md"),
        os.path.join(repo_root, "docs", "**", "*SKILL*.md"),
    ]

    found = set()
    for pattern in patterns:
        for path in glob.glob(pattern, recursive=True):
            if os.path.isfile(path):
                found.add(os.path.normpath(path))

    # AGENTS.md can reference external/absolute skill paths on local machine
    agents_md = os.path.join(repo_root, "AGENTS.md")
    for path in extract_skill_paths_from_agents_md(agents_md):
        found.add(path)

    return sorted(found)


def skill_to_example(skill_text: str, skill_path: str):
    name = os.path.basename(os.path.dirname(skill_path)) or "unknown_skill"
    text = normalize_ws(skill_text)[:3500]
    return {
        "input": f"Explique objetivamente para que serve a skill {name}, quando usar e quando nao usar.",
        "output": f"Skill {name}: {text}",
        "_skill_name": name,
        "_skill_path": skill_path,
    }


def is_quality_skill_example(ex: dict):
    output = normalize_ws(ex.get("output", ""))
    inp = normalize_ws(ex.get("input", ""))

    if len(inp) < 20 or len(output) < 120:
        return False, "too_short"

    bad_patterns = [
        r"\bTODO\b",
        r"lorem ipsum",
        r"placeholder",
        r"skill converted from",
    ]
    if any(re.search(p, output, flags=re.IGNORECASE) for p in bad_patterns):
        return False, "boilerplate"

    keyword_hits = 0
    quality_keywords = [
        "use when", "do not use", "triggers", "workflow", "guidance",
        "best practice", "security", "deployment", "integration", "validation",
    ]
    low_out = output.lower()
    for kw in quality_keywords:
        if kw in low_out:
            keyword_hits += 1
    if keyword_hits < 1:
        return False, "low_signal"

    return True, "ok"


repo_root = find_repo_root()
skill_files = discover_skill_files(repo_root)

print("Repo root detectado:", repo_root)
print("Skill files encontrados:", len(skill_files))
if skill_files:
    for p in skill_files[:8]:
        print(" -", p)
    if len(skill_files) > 8:
        print(f" ... (+{len(skill_files)-8} arquivos)")

# Coleta de skills
skill_examples_raw = []
skill_rejections = Counter()

for path in skill_files:
    try:
        with open(path, "r", encoding="utf-8", errors="ignore") as f:
            content = f.read()
        if normalize_ws(content):
            skill_examples_raw.append(skill_to_example(content, path))
        else:
            skill_rejections["empty_file"] += 1
    except Exception:
        skill_rejections["read_error"] += 1

# Filtro de qualidade
skill_examples_filtered = []
seen_pairs = set()

for ex in skill_examples_raw:
    ok, reason = is_quality_skill_example(ex)
    if not ok:
        skill_rejections[reason] += 1
        continue

    pair = (normalize_ws(ex["input"]).lower(), normalize_ws(ex["output"]).lower())
    if pair in seen_pairs:
        skill_rejections["duplicate"] += 1
        continue
    seen_pairs.add(pair)

    clean_ex = {
        "input": ex["input"],
        "output": ex["output"],
        "source": "skill",
        "category": "apex_skills",
        "skill_name": ex.get("_skill_name", "unknown"),
        "skill_path": ex.get("_skill_path", ""),
    }
    skill_examples_filtered.append(clean_ex)

print("Skill examples (raw):", len(skill_examples_raw))
print("Skill examples (filtered):", len(skill_examples_filtered))
print("Skill rejection summary:", dict(skill_rejections))

USE_SKILL_AUGMENT = True
if USE_SKILL_AUGMENT and skill_examples_filtered:
    train_raw = train_raw + skill_examples_filtered
    print("Treino apos augment skills filtradas:", len(train_raw))
else:
    print("Aviso: nenhum exemplo de skill aprovado para augment.")

## 6) Carregar Gemma 2 2B IT (base aberta)
Sem token obrigatorio.

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

MODEL_CANDIDATES = [
    'unsloth/gemma-2-2b-it',
    'unsloth/gemma-2-2b-it-bnb-4bit'
]

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

model = None
tokenizer = None
model_name = None
last_err = None

for candidate in MODEL_CANDIDATES:
    try:
        print('Tentando:', candidate)
        tokenizer = AutoTokenizer.from_pretrained(candidate)
        if tokenizer.pad_token is None:
            tokenizer.pad_token = tokenizer.eos_token
        tokenizer.padding_side = 'right'

        model = AutoModelForCausalLM.from_pretrained(
            candidate,
            quantization_config=bnb_config,
            device_map='auto',
            torch_dtype=torch.bfloat16
        )
        model_name = candidate
        break
    except Exception as e:
        last_err = e
        print('Falhou:', str(e)[:160])

if model is None:
    raise RuntimeError(f'Nao foi possivel carregar base aberta. Ultimo erro: {last_err}')

print('Modelo carregado:', model_name)

## 7) Configurar LoRA

In [ ]:
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training, TaskType

lora_config = LoraConfig(
    r=8,
    lora_alpha=32,
    target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj'],
    lora_dropout=0.05,
    bias='none',
    task_type=TaskType.CAUSAL_LM
)

model = prepare_model_for_kbit_training(model)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

## 8) Preparar dataset instruct

In [ ]:
from datasets import Dataset

def to_input_output(ex):
    if 'input' in ex and 'output' in ex:
        return ex['input'], ex['output']
    msgs = ex.get('messages', [])
    user_txt = ''
    asst_txt = ''
    for m in msgs:
        role = str(m.get('role', '')).lower()
        content = str(m.get('content', ''))
        if role == 'user' and not user_txt:
            user_txt = content
        elif role == 'assistant' and not asst_txt:
            asst_txt = content
    return user_txt, asst_txt

def format_example(ex):
    user_text, model_text = to_input_output(ex)
    return {
        'text': (
            '<start_of_turn>user\n' + user_text + '<end_of_turn>\n' +
            '<start_of_turn>model\n' + model_text + '<end_of_turn>'
        )
    }

train_formatted = [format_example(ex) for ex in train_raw if to_input_output(ex)[0] and to_input_output(ex)[1]]
train_dataset = Dataset.from_list(train_formatted)
split = train_dataset.train_test_split(test_size=0.1, seed=42)
train_dataset = split['train']
eval_dataset = split['test']

print('Train:', len(train_dataset), 'Eval:', len(eval_dataset), 'Test holdout:', len(test_raw))

## 9) Tokenizacao + treino

In [ ]:
from transformers import TrainingArguments, Trainer, DataCollatorForLanguageModeling

def tokenize_function(examples):
    tokens = tokenizer(
        examples['text'],
        truncation=True,
        padding='max_length',
        max_length=512
    )
    tokens['labels'] = tokens['input_ids'].copy()
    return tokens

tokenized_train = train_dataset.map(tokenize_function, batched=True, remove_columns=['text'])
tokenized_eval = eval_dataset.map(tokenize_function, batched=True, remove_columns=['text'])

training_args = TrainingArguments(
    output_dir='./gemma-apex-finetuned',
    num_train_epochs=3,
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    gradient_accumulation_steps=4,
    learning_rate=2e-5,
    warmup_steps=10,
    logging_steps=10,
    eval_strategy='steps',
    eval_steps=50,
    save_steps=100,
    save_total_limit=2,
    fp16=True,
    bf16=False,
    gradient_checkpointing=True,
    optim='paged_adamw_8bit',
    report_to='none',
    push_to_hub=False,
    remove_unused_columns=False,
    dataloader_pin_memory=False
)

data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_eval,
    data_collator=data_collator
)

trainer.train()

## 10) Avaliacao em teste separado
Mede qualidade em perguntas nao vistas durante treino.

In [ ]:
import math

def quick_generate(prompt, max_new_tokens=200):
    text = '<start_of_turn>user\n' + prompt + '<end_of_turn>\n<start_of_turn>model\n'
    inputs = tokenizer(text, return_tensors='pt').to(model.device)
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=True, temperature=0.7, top_p=0.95)
    decoded = tokenizer.decode(out[0], skip_special_tokens=True)
    return decoded

sample_eval = test_raw[:10]
for i, ex in enumerate(sample_eval, 1):
    q = ex.get('input') or ''
    if not q:
        continue
    pred = quick_generate(q, max_new_tokens=160)
    print('=' * 80)
    print(f'[{i}] Q:', q[:180])
    print('PRED:', pred[:500])

## 11) Merge LoRA + converter para GGUF

In [ ]:
import os

print('Fundindo LoRA...')
merged_model = model.merge_and_unload()
merged_dir = './gemma-apex-merged'
merged_model.save_pretrained(merged_dir, safe_serialization=True)
tokenizer.save_pretrained(merged_dir)

if not os.path.exists('llama.cpp'):
    os.system('git clone --depth 1 https://github.com/ggerganov/llama.cpp')

os.system('pip install -q -r llama.cpp/requirements.txt gguf sentencepiece protobuf')

GGUF_FP16 = 'apex-ai-f16.gguf'
GGUF_Q4 = 'apex-ai.gguf'

os.system(f'python llama.cpp/convert_hf_to_gguf.py {merged_dir} --outfile {GGUF_FP16} --outtype f16')

os.system('cmake -S llama.cpp -B llama.cpp/build -DGGML_CUDA=OFF >/dev/null 2>&1')
os.system('cmake --build llama.cpp/build --target llama-quantize -j 4 >/dev/null 2>&1')

quant_bin = 'llama.cpp/build/bin/llama-quantize'
if not os.path.exists(quant_bin):
    quant_bin = 'llama.cpp/build/llama-quantize'

if os.path.exists(quant_bin) and os.path.exists(GGUF_FP16):
    os.system(f'{quant_bin} {GGUF_FP16} {GGUF_Q4} Q4_K_M')

final_gguf = GGUF_Q4 if os.path.exists(GGUF_Q4) else GGUF_FP16
print('GGUF final:', final_gguf, 'size MB:', round(os.path.getsize(final_gguf) / 1e6, 1) if os.path.exists(final_gguf) else 'N/A')

## 12) Gerar Modelfile para Ollama

In [ ]:
final_gguf = 'apex-ai.gguf' if os.path.exists('apex-ai.gguf') else 'apex-ai-f16.gguf'
modelfile = f'''FROM ./{final_gguf}
TEMPLATE """<start_of_turn>user
{{{{ .Prompt }}}}<end_of_turn>
<start_of_turn>model
"""
SYSTEM "Voce e a Apex AI, plataforma profissional de arquitetura, construcao, BIM, orcamentos, marketing e gestao. Responda em portugues, de forma tecnica e direta, sem inventar dados ou integracoes que nao existem."
PARAMETER temperature 0.7
PARAMETER top_p 0.95
PARAMETER stop "<end_of_turn>"
'''
with open('Modelfile', 'w', encoding='utf-8') as f:
    f.write(modelfile)
print('Modelfile criado.')

## 13) Download artefatos
Baixe:
- apex-ai.gguf (ou apex-ai-f16.gguf)
- Modelfile

In [ ]:
try:
    from google.colab import files
    if os.path.exists('apex-ai.gguf'):
        files.download('apex-ai.gguf')
    elif os.path.exists('apex-ai-f16.gguf'):
        files.download('apex-ai-f16.gguf')
    files.download('Modelfile')
except Exception as e:
    print('Download automatico indisponivel neste ambiente:', e)

## 14) Uso local no Ollama
No terminal local:

```bash
ollama create apex-ai -f Modelfile
ollama run apex-ai "Quem e voce?"
```

API local:

```http
POST http://localhost:11434/api/chat
Content-Type: application/json

{
  "model": "apex-ai",
  "messages": [{"role":"user","content":"O que e a Apex AI?"}]
}
```

## 15) Trainings extras (A/B profiles)

Esta seção cria perfis de treino para comparar qualidade x custo/tempo:
- `fast_debug`: iteração rápida para validar pipeline.
- `standard_prod`: perfil recomendado para produção.
- `high_quality`: maior qualidade (mais custo/tempo).

Defina `RUN_EXPERIMENTS = True` para executar todos os perfis automaticamente.

In [ ]:
from copy import deepcopy

training_profiles = {
    'fast_debug': {
        'num_train_epochs': 1,
        'learning_rate': 3e-5,
        'per_device_train_batch_size': 2,
        'gradient_accumulation_steps': 2,
        'max_length': 384,
    },
    'standard_prod': {
        'num_train_epochs': 3,
        'learning_rate': 2e-5,
        'per_device_train_batch_size': 2,
        'gradient_accumulation_steps': 4,
        'max_length': 512,
    },
    'high_quality': {
        'num_train_epochs': 4,
        'learning_rate': 1.5e-5,
        'per_device_train_batch_size': 2,
        'gradient_accumulation_steps': 6,
        'max_length': 768,
    }
}

RUN_EXPERIMENTS = False

print('Perfis disponiveis:', list(training_profiles.keys()))
if not RUN_EXPERIMENTS:
    print('RUN_EXPERIMENTS=False -> nenhuma execucao iniciada (modo seguro).')

# Exemplo de template de execucao por perfil (descomente para uso real):
# for profile_name, cfg in training_profiles.items():
#     print(f'Iniciando perfil: {profile_name}')
#     _args = deepcopy(training_args)
#     _args.num_train_epochs = cfg['num_train_epochs']
#     _args.learning_rate = cfg['learning_rate']
#     _args.per_device_train_batch_size = cfg['per_device_train_batch_size']
#     _args.gradient_accumulation_steps = cfg['gradient_accumulation_steps']
#     trainer = Trainer(
#         model=model,
#         args=_args,
#         train_dataset=tokenized_train,
#         eval_dataset=tokenized_eval,
#         data_collator=data_collator,
#     )
#     trainer.train()
#     trainer.save_model(f'./runs/{profile_name}')

## 16) Tests automatizados (sanity + qualidade + regressao)

Esta seção valida:
1. `sanity`: o modelo responde sem erro.
2. `quality`: similaridade básica com teste holdout.
3. `regression`: compara baseline x novo modelo em prompts críticos.

Resultados são exportados para `reports/training_test_report.md`.

In [ ]:
import os
import re
from collections import Counter, defaultdict

os.makedirs("reports", exist_ok=True)


def normalize_text(s: str) -> str:
    s = (s or "").lower().strip()
    s = re.sub(r"\s+", " ", s)
    return s


def tokenize(s: str):
    s = normalize_text(s)
    if not s:
        return []
    return re.findall(r"\w+", s, flags=re.UNICODE)


def token_f1(pred: str, ref: str) -> float:
    p = tokenize(pred)
    r = tokenize(ref)
    if not p or not r:
        return 0.0
    cp, cr = Counter(p), Counter(r)
    inter = sum((cp & cr).values())
    prec = inter / max(len(p), 1)
    rec = inter / max(len(r), 1)
    if prec + rec == 0:
        return 0.0
    return 2 * prec * rec / (prec + rec)


def bleu1(pred: str, ref: str) -> float:
    p = tokenize(pred)
    r = tokenize(ref)
    if not p or not r:
        return 0.0
    cp, cr = Counter(p), Counter(r)
    overlap = sum(min(cp[t], cr[t]) for t in cp)
    return overlap / max(len(p), 1)


def lcs_len(a, b):
    # DP simples para ROUGE-L em amostras curtas
    m, n = len(a), len(b)
    dp = [[0] * (n + 1) for _ in range(m + 1)]
    for i in range(1, m + 1):
        for j in range(1, n + 1):
            if a[i - 1] == b[j - 1]:
                dp[i][j] = dp[i - 1][j - 1] + 1
            else:
                dp[i][j] = max(dp[i - 1][j], dp[i][j - 1])
    return dp[m][n]


def rouge_l_f1(pred: str, ref: str) -> float:
    p = tokenize(pred)
    r = tokenize(ref)
    if not p or not r:
        return 0.0
    lcs = lcs_len(p, r)
    prec = lcs / max(len(p), 1)
    rec = lcs / max(len(r), 1)
    if prec + rec == 0:
        return 0.0
    return 2 * prec * rec / (prec + rec)


def extract_test_pair(ex):
    if "input" in ex and "output" in ex:
        return ex["input"], ex["output"]
    msgs = ex.get("messages", [])
    user_txt, asst_txt = "", ""
    for m in msgs:
        role = str(m.get("role", "")).lower()
        content = str(m.get("content", ""))
        if role == "user" and not user_txt:
            user_txt = content
        elif role == "assistant" and not asst_txt:
            asst_txt = content
    return user_txt, asst_txt


def categorize_question(q: str):
    t = normalize_text(q)
    if re.search(r"\b(bim|ifc|revit|clash|modelo 3d)\b", t):
        return "bim"
    if re.search(r"\b(orcamento|custo|budget|finance|caixa)\b", t):
        return "finance"
    if re.search(r"\b(marketing|campanha|social|lead|crm)\b", t):
        return "marketing_crm"
    if re.search(r"\b(contrato|juridico|compliance|nr|norma|risco)\b", t):
        return "compliance_legal"
    if re.search(r"\b(api|deploy|servidor|mcp|integracao|local|ollama|gguf)\b", t):
        return "platform_ops"
    return "general"


# 1) Sanity checks
sanity_prompts = [
    "Quem e voce?",
    "Explique em 3 linhas o que a Apex AI faz.",
    "Como usar apex-local com Ollama?",
]

sanity_results = []
for prompt in sanity_prompts:
    try:
        out = quick_generate(prompt, max_new_tokens=120)
        ok = bool(out and len(out.strip()) > 0)
        sanity_results.append((prompt, ok, out[:240]))
    except Exception as e:
        sanity_results.append((prompt, False, str(e)))

sanity_pass = all(x[1] for x in sanity_results)
print("SANITY PASS:", sanity_pass)

# 2) Quality + BLEU/ROUGE + category eval
sample_holdout = test_raw[: min(80, len(test_raw))]
rows = []
by_cat = defaultdict(list)

for ex in sample_holdout:
    q, ref = extract_test_pair(ex)
    if not q or not ref:
        continue

    pred = quick_generate(q, max_new_tokens=180)
    f1 = token_f1(pred, ref)
    b1 = bleu1(pred, ref)
    rL = rouge_l_f1(pred, ref)
    cat = categorize_question(q)

    row = {
        "question": q[:160],
        "category": cat,
        "token_f1": f1,
        "bleu1": b1,
        "rouge_l_f1": rL,
        "pred_preview": pred[:220].replace("\n", " "),
    }
    rows.append(row)
    by_cat[cat].append(row)

if rows:
    avg_f1 = sum(r["token_f1"] for r in rows) / len(rows)
    avg_bleu1 = sum(r["bleu1"] for r in rows) / len(rows)
    avg_rouge_l = sum(r["rouge_l_f1"] for r in rows) / len(rows)
else:
    avg_f1 = avg_bleu1 = avg_rouge_l = 0.0

print("QUALITY n:", len(rows))
print("AVG token-F1:", round(avg_f1, 4))
print("AVG BLEU-1:", round(avg_bleu1, 4))
print("AVG ROUGE-L(F1):", round(avg_rouge_l, 4))

# 3) Regression prompts
regression_prompts = [
    "Quais os passos para usar apex-local no chat?",
    "Como transformar fine-tuning em GGUF?",
    "Qual a diferenca entre treino e teste holdout?",
]
regression_outputs = []
for p in regression_prompts:
    regression_outputs.append({
        "prompt": p,
        "prediction": quick_generate(p, max_new_tokens=180)[:500],
    })

# 4) Export markdown report (versionavel no repo)
report_lines = []
report_lines.append("# Apex AI Training + Test Report")
report_lines.append("")
report_lines.append("## Summary")
report_lines.append(f"- Sanity pass: **{sanity_pass}**")
report_lines.append(f"- Quality sample size: **{len(rows)}**")
report_lines.append(f"- AVG token-F1: **{avg_f1:.4f}**")
report_lines.append(f"- AVG BLEU-1: **{avg_bleu1:.4f}**")
report_lines.append(f"- AVG ROUGE-L(F1): **{avg_rouge_l:.4f}**")
report_lines.append("")

report_lines.append("## Evaluation by category")
for cat in sorted(by_cat.keys()):
    cat_rows = by_cat[cat]
    c_f1 = sum(r["token_f1"] for r in cat_rows) / len(cat_rows)
    c_b1 = sum(r["bleu1"] for r in cat_rows) / len(cat_rows)
    c_rl = sum(r["rouge_l_f1"] for r in cat_rows) / len(cat_rows)
    report_lines.append(f"- **{cat}**: n={len(cat_rows)}, token-F1={c_f1:.4f}, BLEU-1={c_b1:.4f}, ROUGE-L(F1)={c_rl:.4f}")

report_lines.append("")
report_lines.append("## Sanity checks")
for p, ok, preview in sanity_results:
    report_lines.append(f"- [{'OK' if ok else 'FAIL'}] {p}")
    report_lines.append(f"  - Preview: {preview.replace(chr(10), ' ')}")

report_lines.append("")
report_lines.append("## Regression prompts")
for row in regression_outputs:
    report_lines.append(f"- Prompt: {row['prompt']}")
    report_lines.append(f"  - Output: {row['prediction'].replace(chr(10), ' ')}")

report_lines.append("")
report_lines.append("## Top holdout samples (preview)")
for r in rows[:10]:
    report_lines.append(f"- [{r['category']}] F1={r['token_f1']:.3f} BLEU1={r['bleu1']:.3f} ROUGE-L={r['rouge_l_f1']:.3f} | Q: {r['question']}")

report_path = "reports/training_test_report.md"
with open(report_path, "w", encoding="utf-8") as f:
    f.write("\n".join(report_lines))

print("Relatorio salvo em:", report_path)
print("\n".join(report_lines[:14]))